# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Find the starter CSV from common relative locations so the notebook runs from different CWDs
candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("..", "data", "raw", "content_refresh_anonymized.csv"),
    Path("..", "..", "data", "raw", "content_refresh_anonymized.csv"),
]
file_path = None
for candidate in candidates:
    if candidate.exists():
        file_path = candidate.resolve()
        break
if file_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv from the notebook working directory. ",
        f"Tried: {[str(p) for p in candidates]}"
    )

print(f"Loading data from: {file_path}")
df = pd.read_csv(file_path)

# Unit of analysis and time window claim
unit_of_analysis = 'one row = one pseudonymized content item (page) aggregated over the recall window encoded by the *_90d and *_30d metrics'
time_window = 'primary window: 90 days (impressions_90d); auxiliary 30-day windows present: impressions_last_30d, impressions_prev_30d'

print(unit_of_analysis)
print(time_window)
print("Dataset shape:", df.shape)
print("Unique content_id:", df['content_id'].nunique())
print("Unique client_id:", df['client_id'].nunique())
print(df.head(5))

Loading data from: /home/otto/Documents/projects/flyrank-internship/data/raw/content_refresh_anonymized.csv
one row = one pseudonymized content item (page) aggregated over the recall window encoded by the *_90d and *_30d metrics
primary window: 90 days (impressions_90d); auxiliary 30-day windows present: impressions_last_30d, impressions_prev_30d
Dataset shape: (30000, 44)
Unique content_id: 30000
Unique client_id: 32
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional   

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
# Fields: sort into feature / label / context / excluded buckets
features = [
    'impressions_90d', 'impressions_last_30d', 'impressions_prev_30d',
    'ctr', 'avg_position', 'content_type', 'title_length', 'body_length'
]
label = 'is_declining_label'
context = [
    'content_id', 'client_id', 'keyword_id', 'trend_direction', 'trend_pct'
]
excluded = {
    'trend_direction': 'Excluded as a model feature (leakage) because it directly encodes the downstream outcome used to create the label.',
    'trend_pct': 'Excluded for the same reason and because it summarizes future trend magnitude; including it would leak information.'
}
print('\nFeatures:')
print(features)
print('\nLabel:')
print(label)
print('\nContext (kept for joins / provenance):')
print(context)
print('\nExcluded (with reasons):')
for k,v in excluded.items():
    print(f'{k}: {v}')


Features:
['impressions_90d', 'impressions_last_30d', 'impressions_prev_30d', 'ctr', 'avg_position', 'content_type', 'title_length', 'body_length']

Label:
is_declining_label

Context (kept for joins / provenance):
['content_id', 'client_id', 'keyword_id', 'trend_direction', 'trend_pct']

Excluded (with reasons):
trend_direction: Excluded as a model feature (leakage) because it directly encodes the downstream outcome used to create the label.
trend_pct: Excluded for the same reason and because it summarizes future trend magnitude; including it would leak information.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# Verification queries: grain, counts, missing values, windows
if 'df' not in globals():
    raise RuntimeError("The dataframe 'df' is not defined. Run the unit-of-analysis cell first.")

print('Columns in dataframe:')
print(list(df.columns))

# Grain check: how many rows per content_id? ideally 1 per content item in this export
row_per_content = df.groupby('content_id').size().describe()
print('\nRows per content_id (describe):')
print(row_per_content)

# Desired fields to inspect
desired_cols = ['is_declining_label', 'trend_direction', 'avg_position', 'impressions_90d', 'ctr']
present = [c for c in desired_cols if c in df.columns]
missing_cols = [c for c in desired_cols if c not in df.columns]
if missing_cols:
    print(f"\nWarning: these expected columns are missing from the dataframe: {missing_cols}")

# If label missing but trend_direction exists, derive a temporary label for verification only
df_check = df
if 'is_declining_label' not in df_check.columns and 'trend_direction' in df_check.columns:
    print("Deriving temporary 'is_declining_label' from 'trend_direction' for verification (trend_direction=='down').")
    df_check = df_check.copy()
    df_check['is_declining_label'] = df_check['trend_direction'].str.lower().eq('down')
    if 'is_declining_label' not in present:
        present.append('is_declining_label')

# Missingness for available important fields
if present:
    missing = df_check[present].isna().sum()
    print('\nMissing counts (for available fields):')
    print(missing)
else:
    print('\nNo desired fields present to compute missingness.')

# Label distribution (use derived label if original missing)
if 'is_declining_label' in df_check.columns:
    print('\nLabel distribution (value_counts):')
    print(df_check['is_declining_label'].value_counts(dropna=False))
else:
    print('\nNo label available to show distribution.')

# avg_position special handling: zeros indicate no data in data dictionary
if 'avg_position' in df.columns:
    zero_pos = (df['avg_position'] == 0).sum()
    print(f"\navg_position == 0 (no-data) count: {zero_pos} / {len(df)}")
else:
    print('\navg_position column not present.')

# Check 30d windows presence vs 90d
cols_windows = [c for c in ['impressions_90d','impressions_last_30d','impressions_prev_30d'] if c in df.columns]
if cols_windows:
    print('\nImpressions windows summary:')
    print(df[cols_windows].describe().transpose())
else:
    print('\nNo impressions window columns found to summarize.')

Columns in dataframe:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Rows per content_id (describe):
count    30000.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64

Deriving temporary 'is

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [6]:
# Data limits: explain what this dataset cannot tell and quantify a couple of limits
limits = []
limits.append(
    ("No causal claims",
     "This observational export (search console-like metrics) cannot attribute why traffic changed — only that it changed. Correlation not causation.")
)
limits.append(
    ("GSC-only coverage and missing keywords",
     "Metrics come from search console-like sources and may miss external referral or non-search traffic; keyword coverage may be incomplete.")
)
limits.append(
    ("Zero-position rows are not real ranks",
     "Rows with avg_position == 0 encode no rank data; they were left in the export and should be treated as missing for ranking features.")
)
# Quantify one limit: fraction rows with impressions_90d == 0
zero_impr_frac = (df['impressions_90d'] == 0).mean()
print('Data limits (qualitative):')
for title,desc in limits:
    print(f'- {title}: {desc}')

print(f'\nFraction rows with impressions_90d == 0: {zero_impr_frac:.3f}')

# Suggest next steps for data hygiene
print('\nSuggested hygiene steps: replace avg_position == 0 with NaN before modeling; treat trend_direction/trend_pct as label construction only; aggregate duplicate rows if grain requires it.')

Data limits (qualitative):
- No causal claims: This observational export (search console-like metrics) cannot attribute why traffic changed — only that it changed. Correlation not causation.
- GSC-only coverage and missing keywords: Metrics come from search console-like sources and may miss external referral or non-search traffic; keyword coverage may be incomplete.
- Zero-position rows are not real ranks: Rows with avg_position == 0 encode no rank data; they were left in the export and should be treated as missing for ranking features.

Fraction rows with impressions_90d == 0: 0.000

Suggested hygiene steps: replace avg_position == 0 with NaN before modeling; treat trend_direction/trend_pct as label construction only; aggregate duplicate rows if grain requires it.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.